# FASE 4: Preparación de Datos para Modelización

**Objetivo:** Transformar variables para crear un tablón listo para modelización
**Responsable:** Agente A_04_PreparadorDatos
**Fecha inicio:** 2026-06-04

## FASE PREVIA: Carga del Dataframe y Contexto

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataframe desde el archivo POST-EDA
df = pd.read_pickle('../02_datos/03_Entrenamiento/03_train_tablon_eda.pkl')
print(f'✓ Dataframe cargado: {df.shape[0]} registros × {df.shape[1]} columnas')
print(f'✓ Nulos totales: {df.isnull().sum().sum()}')

✓ Dataframe cargado: 6279 registros × 21 columnas
✓ Nulos totales: 0


In [12]:
df.head()

,id,origen,fuente,no_enviar_email,compra,visitas_total,tiempo_en_site_total,paginas_vistas_visita,ult_actividad,ambito,...,conociste_google,conociste_periodico,conociste_facebook,conociste_referencias,score_actividad,score_perfil,descarga_lm,tiene_score_actividad,tiene_score_perfil,usuario_nuevo
1242,646181,API,Google,No,0,2.0,200,2.0,Email Opened,Select,...,No,No,No,No,13.0,15.0,No,1,1,0
8583,582361,Landing Page Submission,Direct Traffic,No,0,4.0,373,4.0,Had a Phone Conversation,Human Resource Management,...,No,No,No,No,13.0,20.0,Yes,1,1,0
5308,607963,Landing Page Submission,Direct Traffic,No,0,2.0,1122,2.0,Email Opened,IT Projects Management,...,No,No,No,No,0.0,0.0,Yes,0,0,1
8030,586622,API,Chat,No,0,0.0,0,0.0,Chat Conversation,Not Specified,...,No,No,No,No,17.0,15.0,No,1,1,0
2287,636823,Lead Add Form,Reference,No,1,0.0,0,0.0,SMS Sent,Marketing Management,...,No,No,No,No,15.0,20.0,No,1,1,0


In [2]:
# Mostrar estructura básica del dataframe
print("Estructura del dataframe:")
df.info()

Estructura del dataframe:
<class 'pandas.DataFrame'>
Index: 6279 entries, 1242 to 1607
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     6279 non-null   int64  
 1   origen                 6279 non-null   str    
 2   fuente                 6279 non-null   str    
 3   no_enviar_email        6279 non-null   str    
 4   compra                 6279 non-null   int64  
 5   visitas_total          6279 non-null   float64
 6   tiempo_en_site_total   6279 non-null   int64  
 7   paginas_vistas_visita  6279 non-null   float64
 8   ult_actividad          6279 non-null   str    
 9   ambito                 6279 non-null   str    
 10  ocupacion              6279 non-null   str    
 11  conociste_google       6279 non-null   str    
 12  conociste_periodico    6279 non-null   str    
 13  conociste_facebook     6279 non-null   str    
 14  conociste_referencias  6279 non-null   str 

## FASE 1: Diagnóstico Automático de Variables

In [3]:
# Diagnóstico automático de variables
diagnostico = []

for col in df.columns:
    diagnostico.append({
        'Variable': col,
        'Tipo': str(df[col].dtype),
        'Cardinalidad': df[col].nunique(),
        'Missing_%': round(100 * df[col].isnull().sum() / len(df), 2),
    })

df_diag = pd.DataFrame(diagnostico)

# Clasificar variables por tipo
print("="*100)
print("DIAGNÓSTICO AUTOMÁTICO DE VARIABLES")
print("="*100)
print(f"\nDataframe: {df.shape[0]} registros × {df.shape[1]} columnas\n")
print(df_diag.to_string(index=False))

DIAGNÓSTICO AUTOMÁTICO DE VARIABLES

Dataframe: 6279 registros × 21 columnas

             Variable    Tipo  Cardinalidad  Missing_%
                   id   int64          6279        0.0
               origen     str             4        0.0
               fuente     str            17        0.0
      no_enviar_email     str             2        0.0
               compra   int64             2        0.0
        visitas_total float64            40        0.0
 tiempo_en_site_total   int64          1555        0.0
paginas_vistas_visita float64            98        0.0
        ult_actividad     str            16        0.0
               ambito     str            20        0.0
            ocupacion     str             7        0.0
     conociste_google     str             2        0.0
  conociste_periodico     str             2        0.0
   conociste_facebook     str             2        0.0
conociste_referencias     str             2        0.0
      score_actividad float64            1

In [5]:
# Clasificar variables por tipo
print("\n" + "="*100)
print("CLASIFICACIÓN DE VARIABLES POR TIPO")
print("="*100)

# Identificar tipos
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Separar numéricas en continuas vs discretas
numeric_continuas = []
numeric_discretas = []
for col in numeric_cols:
    n_unique = df[col].nunique()
    if n_unique > 20:
        numeric_continuas.append(col)
    else:
        numeric_discretas.append(col)

# Identificar binarias
binarias = [col for col in categorical_cols if df[col].nunique() <= 2]
categoricas_nominales = [col for col in categorical_cols if col not in binarias and df[col].nunique() <= 15]
alta_cardinalidad = [col for col in categorical_cols if df[col].nunique() > 15]

# Mostrar clasificación
print(f"\n📊 NUMÉRICAS CONTINUAS ({len(numeric_continuas)}):")
for col in numeric_continuas:
    print(f"  • {col}: {df[col].nunique()} valores únicos")

print(f"\n📊 NUMÉRICAS DISCRETAS ({len(numeric_discretas)}):")
for col in numeric_discretas:
    print(f"  • {col}: {df[col].nunique()} valores únicos")

print(f"\n📊 CATEGÓRICAS BINARIAS ({len(binarias)}):")
for col in binarias:
    vals = df[col].value_counts().to_dict()
    print(f"  • {col}: {vals}")

print(f"\n📊 CATEGÓRICAS NOMINALES ({len(categoricas_nominales)}):")
for col in categoricas_nominales:
    print(f"  • {col}: {df[col].nunique()} valores únicos")

print(f"\n📊 ALTA CARDINALIDAD ({len(alta_cardinalidad)}):")
for col in alta_cardinalidad:
    print(f"  • {col}: {df[col].nunique()} valores únicos")

print(f"\n✓ Total: {len(numeric_continuas) + len(numeric_discretas) + len(binarias) + len(categoricas_nominales) + len(alta_cardinalidad)} variables")


CLASIFICACIÓN DE VARIABLES POR TIPO

📊 NUMÉRICAS CONTINUAS (4):
  • id: 6279 valores únicos
  • visitas_total: 40 valores únicos
  • tiempo_en_site_total: 1555 valores únicos
  • paginas_vistas_visita: 98 valores únicos

📊 NUMÉRICAS DISCRETAS (6):
  • compra: 2 valores únicos
  • score_actividad: 13 valores únicos
  • score_perfil: 11 valores únicos
  • tiene_score_actividad: 2 valores únicos
  • tiene_score_perfil: 2 valores únicos
  • usuario_nuevo: 2 valores únicos

📊 CATEGÓRICAS BINARIAS (6):
  • no_enviar_email: {'No': 5792, 'Yes': 487}
  • conociste_google: {'No': 6269, 'Yes': 10}
  • conociste_periodico: {'No': 6278, 'Yes': 1}
  • conociste_facebook: {'No': 6276, 'Yes': 3}
  • conociste_referencias: {'No': 6275, 'Yes': 4}
  • descarga_lm: {'No': 4291, 'Yes': 1988}

📊 CATEGÓRICAS NOMINALES (2):
  • origen: 4 valores únicos
  • ocupacion: 7 valores únicos

📊 ALTA CARDINALIDAD (3):
  • fuente: 17 valores únicos
  • ult_actividad: 16 valores únicos
  • ambito: 20 valores únicos

✓ 

## FASE 2: Análisis de Distribuciones para Transformaciones

In [6]:
# Analizar distribuciones de variables numéricas
from scipy.stats import skew, kurtosis

print("\n" + "="*100)
print("ANÁLISIS DE DISTRIBUCIONES (VARIABLES NUMÉRICAS)")
print("="*100 + "\n")

for col in numeric_continuas:
    if col != 'id':  # Excluir ID
        sk = skew(df[col].dropna())
        kurt = kurtosis(df[col].dropna())
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        outliers = ((df[col] < (q1 - 1.5*iqr)) | (df[col] > (q3 + 1.5*iqr))).sum()
        pct_outliers = round(100 * outliers / len(df), 2)
        
        print(f"📊 {col}:")
        print(f"   Skewness: {sk:.2f} | Kurtosis: {kurt:.2f} | Outliers: {pct_outliers}%")
        print(f"   Min: {df[col].min():.2f} | Max: {df[col].max():.2f} | Mean: {df[col].mean():.2f}")
        
        # Recomendar transformación
        if abs(sk) > 2:
            print(f"   ⚠️  ALTAMENTE SESGADA → Recomendar LOG o BOX-COX")
        elif abs(sk) > 1:
            print(f"   ⚠️  Moderadamente sesgada → Recomendar LOG o transformación estadística")
        print()


ANÁLISIS DE DISTRIBUCIONES (VARIABLES NUMÉRICAS)

📊 visitas_total:
   Skewness: 21.98 | Kurtosis: 925.89 | Outliers: 2.99%
   Min: 0.00 | Max: 251.00 | Mean: 3.50
   ⚠️  ALTAMENTE SESGADA → Recomendar LOG o BOX-COX

📊 tiempo_en_site_total:
   Skewness: 0.96 | Kurtosis: -0.39 | Outliers: 0.0%
   Min: 0.00 | Max: 2272.00 | Mean: 488.32

📊 paginas_vistas_visita:
   Skewness: 3.39 | Kurtosis: 56.30 | Outliers: 3.85%
   Min: 0.00 | Max: 55.00 | Mean: 2.40
   ⚠️  ALTAMENTE SESGADA → Recomendar LOG o BOX-COX



## FASE 3: Aplicación de Transformaciones (Pipeline Secuencial)

In [7]:
# FASE 1: Transformaciones numéricas según PlantillaTransformaciones.xlsx
print("\n" + "="*100)
print("FASE 1: TRANSFORMACIONES NUMÉRICAS (MINMAX SCALING - SEGÚN PLANTILLA)")
print("="*100 + "\n")

# Listas de tracking
cols_fase1_numericas = []  # Features numéricas escaladas
cols_fase2_binarias = []   # Features binarias (OHE)
cols_finales_df = ['compra']  # Iniciar con target
cols_intermedias_excluir = []  # Columnas intermedias que NO van al df final
cols_aisladas = []  # Columnas a aislar

# Diccionario para almacenar dataframes transformados
dfs_transformados = {}

# SUBBLOQUE 1A: Numéricas con MinMax Scaling (MMS)
# Según PlantillaTransformaciones.xlsx: visitas_total, tiempo_en_site_total, 
# paginas_vistas_visita, score_actividad, score_perfil
print("📊 SUBBLOQUE 1A: Numéricas continuas (MinMax Scaling - MMS)")
print("-" * 100)

vars_minmax = ['visitas_total', 'tiempo_en_site_total', 'paginas_vistas_visita',
               'score_actividad', 'score_perfil']

for col in vars_minmax:
    if col in df.columns:
        # Aplicar MinMaxScaler (rango [0, 1])
        mms = MinMaxScaler()
        df_mms = pd.DataFrame(
            mms.fit_transform(df[[col]]),
            columns=[f'{col}_mms'],
            index=df.index
        )

        dfs_transformados[f'{col}_mms'] = df_mms
        cols_fase1_numericas.append(f'{col}_mms')
        cols_finales_df.append(f'{col}_mms')
        cols_intermedias_excluir.append(col)

        print(f"  ✓ {col} → mms (MinMax Scaling [0,1])")

print(f"\n✓ {len(vars_minmax)} variables numéricas transformadas con MinMax Scaling\n")

# SUBBLOQUE 1B: Variables sin transformación
# Según PlantillaTransformaciones.xlsx: usuario_nuevo
print("📊 SUBBLOQUE 1B: Variables sin transformación")
print("-" * 100)

vars_sin_transformar = ['usuario_nuevo']

for col in vars_sin_transformar:
    if col in df.columns:
        df_notrans = df[[col]].copy()
        dfs_transformados[col] = df_notrans
        cols_fase1_numericas.append(col)
        cols_finales_df.append(col)

        print(f"  ✓ {col}: Sin transformación (formato correcto)")

print(f"\n✓ {len(vars_sin_transformar)} variable sin transformación incluida\n")

# SUBBLOQUE 1C: Variables a aislar (NO van a modelización, se reinsertan después)
# Según PlantillaTransformaciones.xlsx: no_enviar_email, no_llamar, id
print("📊 SUBBLOQUE 1C: Variables a aislar (post-modelización)")
print("-" * 100)

vars_aisladas = ['no_enviar_email', 'id']  # no_llamar no existe en dataset

for col in vars_aisladas:
    if col in df.columns:
        df_aislada = df[[col]].copy()
        dfs_transformados[f'_aislada_{col}'] = df_aislada
        cols_aisladas.append(col)
        cols_intermedias_excluir.append(col)

        print(f"  ✓ {col}: Aislada (se reinsertar después de modelización)")

print(f"\n✓ {len(vars_aisladas)} variables aisladas\n")

print("="*100)
print(f"✓ FASE 1 COMPLETADA")
print(f"  - Features numéricas escaladas (MMS): {len([c for c in cols_fase1_numericas if '_mms' in c])}")
print(f"  - Features sin transformación: {len([c for c in cols_fase1_numericas if '_mms' not in c])}")
print(f"  - Variables aisladas: {len(cols_aisladas)}")
print(f"  - Features finales acumuladas: {len(cols_finales_df)}")
print("="*100)


FASE 1: TRANSFORMACIONES NUMÉRICAS (MINMAX SCALING - SEGÚN PLANTILLA)

📊 SUBBLOQUE 1A: Numéricas continuas (MinMax Scaling - MMS)
----------------------------------------------------------------------------------------------------
  ✓ visitas_total → mms (MinMax Scaling [0,1])
  ✓ tiempo_en_site_total → mms (MinMax Scaling [0,1])
  ✓ paginas_vistas_visita → mms (MinMax Scaling [0,1])
  ✓ score_actividad → mms (MinMax Scaling [0,1])
  ✓ score_perfil → mms (MinMax Scaling [0,1])

✓ 5 variables numéricas transformadas con MinMax Scaling

📊 SUBBLOQUE 1B: Variables sin transformación
----------------------------------------------------------------------------------------------------
  ✓ usuario_nuevo: Sin transformación (formato correcto)

✓ 1 variable sin transformación incluida

📊 SUBBLOQUE 1C: Variables a aislar (post-modelización)
----------------------------------------------------------------------------------------------------
  ✓ no_enviar_email: Aislada (se reinsertar después de m

In [8]:
# FASE 2: Transformaciones categóricas según PlantillaTransformaciones.xlsx
print("\n" + "="*100)
print("FASE 2: TRANSFORMACIONES CATEGÓRICAS (ONE-HOT ENCODING - SEGÚN PLANTILLA)")
print("="*100 + "\n")

# Según PlantillaTransformaciones.xlsx: origen, fuente, ult_actividad, ambito, ocupacion, descarga_lm
# NOTA: NO incluir conociste_google, conociste_periodico, conociste_facebook, conociste_referencias
print("📊 Categóricas nominales (OneHotEncoder drop='first')")
print("-" * 100)

vars_cat_ohe = ['origen', 'fuente', 'ult_actividad', 'ambito', 'ocupacion', 'descarga_lm']

for col in vars_cat_ohe:
    if col in df.columns:
        # OneHotEncoder con drop='first'
        ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=int)
        ohe_array = ohe.fit_transform(df[[col]])

        # Nombres de dummies
        dummies_cols = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]

        df_ohe = pd.DataFrame(ohe_array, columns=dummies_cols, index=df.index)

        for dummy_col in dummies_cols:
            dfs_transformados[dummy_col] = df_ohe[[dummy_col]]
            cols_fase2_binarias.append(dummy_col)
            cols_finales_df.append(dummy_col)

        cols_intermedias_excluir.append(col)
        n_cats = len(ohe.categories_[0])
        print(f"  ✓ {col} → {len(dummies_cols)} dummies (OHE: {n_cats} categorías → {len(dummies_cols)} dummies con drop='first')")

print(f"\n✓ {len(vars_cat_ohe)} variables categóricas transformadas con OneHotEncoding\n")

print("="*100)
print(f"✓ FASE 2 COMPLETADA: {len(cols_fase2_binarias)} features binarias generadas")
print(f"  - Variables categóricas procesadas: {len(vars_cat_ohe)}")
print(f"  - Dummies generados: {len(cols_fase2_binarias)}")
print(f"  - Features finales acumuladas: {len(cols_finales_df)}")
print("="*100)


FASE 2: TRANSFORMACIONES CATEGÓRICAS (ONE-HOT ENCODING - SEGÚN PLANTILLA)

📊 Categóricas nominales (OneHotEncoder drop='first')
----------------------------------------------------------------------------------------------------
  ✓ origen → 3 dummies (OHE: 4 categorías → 3 dummies con drop='first')
  ✓ fuente → 16 dummies (OHE: 17 categorías → 16 dummies con drop='first')
  ✓ ult_actividad → 15 dummies (OHE: 16 categorías → 15 dummies con drop='first')
  ✓ ambito → 19 dummies (OHE: 20 categorías → 19 dummies con drop='first')
  ✓ ocupacion → 6 dummies (OHE: 7 categorías → 6 dummies con drop='first')
  ✓ descarga_lm → 1 dummies (OHE: 2 categorías → 1 dummies con drop='first')

✓ 6 variables categóricas transformadas con OneHotEncoding

✓ FASE 2 COMPLETADA: 60 features binarias generadas
  - Variables categóricas procesadas: 6
  - Dummies generados: 60
  - Features finales acumuladas: 67


In [9]:
# FASE 4: Unión final, manejo de variables aisladas y validaciones
print("\n" + "="*100)
print("FASE 4: UNIÓN FINAL, VARIABLES AISLADAS Y VALIDACIONES")
print("="*100 + "\n")

# 1. Preparar dataframes transformados
print("📋 Paso 1: Preparando dataframes transformados")
print("-" * 100)

lista_dfs = []

# Añadir target (compra) primero
df_target = df[['compra']].copy()
lista_dfs.append(df_target)
print(f"  ✓ Target (compra): {df_target.shape[1]} columna - BINARIO (0/1)")

# Añadir features numéricas transformadas
for col in cols_fase1_numericas:
    if col in dfs_transformados:
        lista_dfs.append(dfs_transformados[col])

print(f"  ✓ Features numéricas transformadas: {len(cols_fase1_numericas)} columnas")

# Añadir features categóricas transformadas (dummies)
for col in cols_fase2_binarias:
    if col in dfs_transformados:
        lista_dfs.append(dfs_transformados[col])

print(f"  ✓ Features categóricas transformadas (dummies): {len(cols_fase2_binarias)} columnas")

# 2. Concatenar horizontalmente
print("\n📋 Paso 2: Concatenando dataframes de modelización")
print("-" * 100)

df_final = pd.concat(lista_dfs, axis=1)
print(f"  ✓ Dataframe de modelización: {df_final.shape[0]} registros × {df_final.shape[1]} columnas")

# 3. Extraer y guardar variables aisladas
print("\n📋 Paso 3: Extrayendo variables aisladas (post-modelización)")
print("-" * 100)

df_aisladas = pd.DataFrame(index=df.index)
for col in cols_aisladas:
    if col in df.columns:
        df_aisladas[col] = df[col].copy()

print(f"  ✓ Variables aisladas extraídas: {df_aisladas.shape[1]} columnas")
print(f"    - {', '.join(cols_aisladas)}")

# 4. Validaciones críticas
print("\n📋 Paso 4: Validaciones de Integridad")
print("-" * 100)

# VALIDACIÓN 1: Número de filas conservado
assert df_final.shape[0] == df.shape[0], "❌ ERROR: Pérdida de filas en unión"
print(f"  ✓ VALIDACIÓN 1: {df_final.shape[0]} filas conservadas")

# VALIDACIÓN 2: Target presente y binario
assert 'compra' in df_final.columns, "❌ ERROR: Target 'compra' no está en df final"
assert set(df_final['compra'].unique()).issubset({0, 1}), "❌ ERROR: Target no es binario"
print(f"  ✓ VALIDACIÓN 2: Target 'compra' presente y binario (0/1)")

# VALIDACIÓN 3: No hay columnas intermedias en df final
intermedias_presentes = set(df_final.columns).intersection(set(cols_intermedias_excluir))
assert len(intermedias_presentes) == 0, f"❌ ERROR: Columnas intermedias en df final: {intermedias_presentes}"
print(f"  ✓ VALIDACIÓN 3: No hay columnas intermedias en df final")

# VALIDACIÓN 4: No hay NaN inesperados
nan_counts = df_final.isnull().sum()
nan_total = nan_counts.sum()
assert nan_total == 0, f"❌ ERROR: Se detectaron {nan_total} NaN en df final"
print(f"  ✓ VALIDACIÓN 4: 0 valores NaN en df final")

# VALIDACIÓN 5: No hay colisiones de nombres
assert len(df_final.columns) == len(set(df_final.columns)), "❌ ERROR: Nombres de columnas duplicados"
print(f"  ✓ VALIDACIÓN 5: Todos los nombres de columnas son únicos ({len(df_final.columns)} columnas)")

# VALIDACIÓN 6: Target está en primera posición
assert df_final.columns[0] == 'compra', "⚠️  ADVERTENCIA: Target no está en primera posición"
print(f"  ✓ VALIDACIÓN 6: Target está en primera posición")

# VALIDACIÓN 7: Variables aisladas están intactas
assert df_aisladas.shape[0] == df.shape[0], "❌ ERROR: Pérdida de filas en variables aisladas"
print(f"  ✓ VALIDACIÓN 7: Variables aisladas conservan {df_aisladas.shape[0]} filas")

# 5. Resumen final
print("\n" + "="*100)
print("✓ FASE 4 COMPLETADA - RESUMEN FINAL")
print("="*100)
print(f"\n📊 DATAFRAME TRANSFORMADO PARA MODELIZACIÓN:")
print(f"  • Registros: {df_final.shape[0]:,}")
print(f"  • Columnas: {df_final.shape[1]}")
print(f"  • Target: 'compra' (BINARIO 0/1, variable objetivo)")
print(f"  • Features numéricas (MinMax Scaling): {len([c for c in cols_fase1_numericas if '_mms' in c])}")
print(f"  • Features categóricas (OHE): {len(cols_fase2_binarias)}")
print(f"  • Total features: {df_final.shape[1] - 1}")
print(f"\n📊 VARIABLES AISLADAS (se reinsertan post-modelización):")
print(f"  • Registros: {df_aisladas.shape[0]:,}")
print(f"  • Variables: {', '.join(cols_aisladas)}")
print(f"\n✓ Preparación de datos completada según PlantillaTransformaciones.xlsx")
print("="*100)


FASE 4: UNIÓN FINAL, VARIABLES AISLADAS Y VALIDACIONES

📋 Paso 1: Preparando dataframes transformados
----------------------------------------------------------------------------------------------------
  ✓ Target (compra): 1 columna - BINARIO (0/1)
  ✓ Features numéricas transformadas: 6 columnas
  ✓ Features categóricas transformadas (dummies): 60 columnas

📋 Paso 2: Concatenando dataframes de modelización
----------------------------------------------------------------------------------------------------
  ✓ Dataframe de modelización: 6279 registros × 67 columnas

📋 Paso 3: Extrayendo variables aisladas (post-modelización)
----------------------------------------------------------------------------------------------------
  ✓ Variables aisladas extraídas: 2 columnas
    - no_enviar_email, id

📋 Paso 4: Validaciones de Integridad
----------------------------------------------------------------------------------------------------
  ✓ VALIDACIÓN 1: 6279 filas conservadas
  ✓ VALIDACIÓ

In [10]:
# FASE 5: Guardado de dataframes transformados
print("\n" + "="*100)
print("FASE 5: GUARDADO DE DATAFRAMES TRANSFORMADOS")
print("="*100 + "\n")

# 1. Guardar el dataframe de modelización
print("📋 Guardando dataframe de modelización")
print("-" * 100)

ruta_modelizacion = '../02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl'
df_final.to_pickle(ruta_modelizacion)

print(f"✓ Dataframe de modelización guardado en: {ruta_modelizacion}")
print(f"  • Registros: {df_final.shape[0]:,}")
print(f"  • Columnas: {df_final.shape[1]}")
print(f"  • Tamaño archivo: {os.path.getsize(ruta_modelizacion) / 1024 / 1024:.2f} MB")

# 2. Guardar variables aisladas
print("\n📋 Guardando variables aisladas (post-modelización)")
print("-" * 100)

ruta_aisladas = '../02_datos/03_Entrenamiento/04_train_variables_aisladas.pkl'
df_aisladas.to_pickle(ruta_aisladas)

print(f"✓ Variables aisladas guardadas en: {ruta_aisladas}")
print(f"  • Registros: {df_aisladas.shape[0]:,}")
print(f"  • Columnas: {df_aisladas.shape[1]}")
print(f"  • Variables: {', '.join(cols_aisladas)}")
print(f"  • Tamaño archivo: {os.path.getsize(ruta_aisladas) / 1024 / 1024:.2f} MB")

# 3. Guardar metadatos de transformación
print("\n📋 Guardando metadatos de transformación")
print("-" * 100)

metadata_transformacion = {
    'fecha_transformacion': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'registros_originales': df.shape[0],
    'registros_finales': df_final.shape[0],
    'variables_numéricas_minmax': [col for col in cols_fase1_numericas if '_mms' in col],
    'variables_sin_transformación': [col for col in cols_fase1_numericas if '_mms' not in col],
    'variables_categóricas_ohe': vars_cat_ohe,
    'dummies_generados': cols_fase2_binarias,
    'variables_aisladas': cols_aisladas,
    'target': 'compra'
}

import json
ruta_metadata = '../02_datos/03_Entrenamiento/04_train_metadata_transformacion.json'
with open(ruta_metadata, 'w') as f:
    json.dump(metadata_transformacion, f, indent=2, ensure_ascii=False)

print(f"✓ Metadatos de transformación guardados en: {ruta_metadata}")

print("\n" + "="*100)
print("✅ FASE 5 COMPLETADA - PREPARACIÓN DE DATOS FINALIZADA")
print("="*100)
print("\n🎯 ARCHIVOS GENERADOS:")
print(f"  1. {ruta_modelizacion}")
print(f"  2. {ruta_aisladas}")
print(f"  3. {ruta_metadata}")
print("\n🎯 PRÓXIMOS PASOS:")
print("  1. Modelización con variables transformadas")
print("  2. Predicciones")
print("  3. Reinserción de variables aisladas en resultados finales")


FASE 5: GUARDADO DE DATAFRAMES TRANSFORMADOS

📋 Guardando dataframe de modelización
----------------------------------------------------------------------------------------------------
✓ Dataframe de modelización guardado en: ../02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl
  • Registros: 6,279
  • Columnas: 67
  • Tamaño archivo: 3.26 MB

📋 Guardando variables aisladas (post-modelización)
----------------------------------------------------------------------------------------------------
✓ Variables aisladas guardadas en: ../02_datos/03_Entrenamiento/04_train_variables_aisladas.pkl
  • Registros: 6,279
  • Columnas: 2
  • Variables: no_enviar_email, id
  • Tamaño archivo: 0.16 MB

📋 Guardando metadatos de transformación
----------------------------------------------------------------------------------------------------
✓ Metadatos de transformación guardados en: ../02_datos/03_Entrenamiento/04_train_metadata_transformacion.json

✅ FASE 5 COMPLETADA - PREPARACIÓN DE DATOS

In [11]:
df_final.head()

,compra,visitas_total_mms,tiempo_en_site_total_mms,paginas_vistas_visita_mms,score_actividad_mms,score_perfil_mms,usuario_nuevo,origen_Landing Page Submission,origen_Lead Add Form,origen_Lead Import,...,ambito_Services Excellence,ambito_Supply Chain Management,ambito_Travel and Tourism,ocupacion_Housewife,ocupacion_Not Provided,ocupacion_Other,ocupacion_Student,ocupacion_Unemployed,ocupacion_Working Professional,descarga_lm_Yes
1242,0,0.007968,0.088028,0.036364,0.722222,0.75,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
8583,0,0.015936,0.164173,0.072727,0.722222,1.00,0,1,0,0,...,0,0,0,0,0,0,0,1,0,1
5308,0,0.007968,0.493838,0.036364,0.000000,0.00,1,1,0,0,...,0,0,0,0,1,0,0,0,0,1
8030,0,0.000000,0.000000,0.000000,0.944444,0.75,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2287,1,0.000000,0.000000,0.000000,0.833333,1.00,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [13]:
df_final.info()


<class 'pandas.DataFrame'>
Index: 6279 entries, 1242 to 1607
Data columns (total 67 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   compra                                      6279 non-null   int64  
 1   visitas_total_mms                           6279 non-null   float64
 2   tiempo_en_site_total_mms                    6279 non-null   float64
 3   paginas_vistas_visita_mms                   6279 non-null   float64
 4   score_actividad_mms                         6279 non-null   float64
 5   score_perfil_mms                            6279 non-null   float64
 6   usuario_nuevo                               6279 non-null   int64  
 7   origen_Landing Page Submission              6279 non-null   int64  
 8   origen_Lead Add Form                        6279 non-null   int64  
 9   origen_Lead Import                          6279 non-null   int64  
 10  fuente_Click2call        